## Part 1: Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [2]:
# Determine the number of unique values in each column
attrition_df.nunique()

,0
Age,43
Attrition,2
BusinessTravel,3
Department,3
DistanceFromHome,29
Education,5
EducationField,6
EnvironmentSatisfaction,4
HourlyRate,71
JobInvolvement,4


In [4]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']].copy()
y_df.head()

,Attrition,Department
0,Yes,Sales
1,No,Research & Development
2,Yes,Research & Development
3,No,Research & Development
4,No,Research & Development


In [5]:
# Create a list of at least 10 column names to use as X data
selected_features = [
    'Age', 'Education', 'DistanceFromHome', 'JobSatisfaction',
    'OverTime', 'StockOptionLevel', 'WorkLifeBalance', 'YearsAtCompany',
    'YearsSinceLastPromotion', 'NumCompaniesWorked'
]

# Create X_df using your selected columns
X_df = attrition_df[selected_features].copy()

# Show the data types for X_df
X_df.dtypes


,0
Age,int64
Education,int64
DistanceFromHome,int64
JobSatisfaction,int64
OverTime,object
StockOptionLevel,int64
WorkLifeBalance,int64
YearsAtCompany,int64
YearsSinceLastPromotion,int64
NumCompaniesWorked,int64


In [11]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_df_encoded, y_df, test_size=0.2, random_state=42)

In [7]:
# Convert your X data to numeric data types however you see fit
# Add new code cells as necessary

X_df['OverTime'].value_counts()


,count
OverTime,
No,1054
Yes,416


In [8]:
# Convert OverTime from Yes/No to 1/0
from sklearn.preprocessing import LabelEncoder

# Create a copy of X_df to avoid modifying the original
X_df_encoded = X_df.copy()

# Convert OverTime: Yes=1, No=0
X_df_encoded['OverTime'] = X_df_encoded['OverTime'].map({'Yes': 1, 'No': 0})

# Verify the conversion worked
print("OverTime conversion:")
print(X_df_encoded['OverTime'].value_counts())
print("\nAll data types after conversion:")
print(X_df_encoded.dtypes)

OverTime conversion:
OverTime
0    1054
1     416
Name: count, dtype: int64

All data types after conversion:
Age                        int64
Education                  int64
DistanceFromHome           int64
JobSatisfaction            int64
OverTime                   int64
StockOptionLevel           int64
WorkLifeBalance            int64
YearsAtCompany             int64
YearsSinceLastPromotion    int64
NumCompaniesWorked         int64
dtype: object


In [12]:
# Create a StandardScaler
scaler = StandardScaler()

# Fit the StandardScaler to the training data and transform training data
X_train_scaled = scaler.fit_transform(X_train)

# Scale the testing data (only transform, don't fit again)
X_test_scaled = scaler.transform(X_test)


In [13]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder for the Department column
dept_encoder = OneHotEncoder(sparse_output=False, drop='first')

# Fit the encoder to the training data
dept_encoder.fit(y_train[['Department']])

# Create two new variables by applying the encoder
# to the training and testing data
y_train_dept_encoded = dept_encoder.transform(y_train[['Department']])
y_test_dept_encoded = dept_encoder.transform(y_test[['Department']])

# Display the result to verify
y_train_dept_encoded


array([[1., 0.],
       [1., 0.],
       [0., 1.],
       ...,
       [1., 0.],
       [1., 0.],
       [0., 1.]])

In [14]:
# Create a OneHotEncoder for the Attrition column
attrition_encoder = OneHotEncoder(sparse_output=False, drop='first')

# Fit the encoder to the training data
attrition_encoder.fit(y_train[['Attrition']])

# Create two new variables by applying the encoder
# to the training and testing data
y_train_attrition_encoded = attrition_encoder.transform(y_train[['Attrition']])
y_test_attrition_encoded = attrition_encoder.transform(y_test[['Attrition']])

# Display the result to verify
y_train_attrition_encoded


array([[0.],
       [0.],
       [0.],
       ...,
       [1.],
       [0.],
       [0.]])

## Part 2: Create, Compile, and Train the Model

In [15]:
# Find the number of columns in the X training data.
input_features = X_train_scaled.shape[1]
print(f"Number of input features: {input_features}")

# Create the input layer
input_layer = layers.Input(shape=(input_features,), name='input')

# Create at least two shared layers
shared1 = layers.Dense(64, activation='relu', name='shared1')(input_layer)
shared2 = layers.Dense(128, activation='relu', name='shared2')(shared1)


Number of input features: 10


In [16]:
# Create a branch for Department
# with a hidden layer and an output layer

# Create the hidden layer
department_hidden = layers.Dense(32, activation='relu', name='department_hidden')(shared2)

# Create the output layer
department_output = layers.Dense(y_train_dept_encoded.shape[1], activation='softmax', name='department_output')(department_hidden)



In [17]:
# Create a branch for Attrition
# with a hidden layer and an output layer

# Create the hidden layer
attrition_hidden = layers.Dense(32, activation='relu', name='attrition_hidden')(shared2)

# Create the output layer
attrition_output = layers.Dense(y_train_attrition_encoded.shape[1], activation='sigmoid', name='attrition_output')(attrition_hidden)


In [18]:
# Create the model
model = Model(inputs=input_layer, outputs=[department_output, attrition_output])

# Compile the model
model.compile(
    optimizer='adam',
    loss={
        'department_output': 'categorical_crossentropy',
        'attrition_output': 'binary_crossentropy'
    },
    metrics={
        'department_output': 'accuracy',
        'attrition_output': 'accuracy'
    }
)

# Summarize the model
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared1 (Dense)     │ (None, 64)        │        704 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared2 (Dense)     │ (None, 128)       │      8,320 │ shared1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_hidden   │ (None, 32)        │      4,128 │ shared2[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attrition_hidden    │ (None, 32)        │      4,128 │ shared2[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_output   │ (None, 2)         │         66 │ department_hidde… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attrition_output    │ (None, 1)         │         33 │ attrition_hidden… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 17,379 (67.89 KB)

 Trainable params: 17,379 (67.89 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# Train the model
history = model.fit(
    X_train_scaled,
    {
        'department_output': y_train_dept_encoded,
        'attrition_output': y_train_attrition_encoded
    },
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - attrition_output_accuracy: 0.7937 - attrition_output_loss: 0.5814 - department_output_accuracy: 0.5361 - department_output_loss: 0.6659 - loss: 1.2477 - val_attrition_output_accuracy: 0.7966 - val_attrition_output_loss: 0.5156 - val_department_output_accuracy: 0.6737 - val_department_output_loss: 0.6248 - val_loss: 1.1582
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - attrition_output_accuracy: 0.8475 - attrition_output_loss: 0.4275 - department_output_accuracy: 0.7229 - department_output_loss: 0.5761 - loss: 1.0038 - val_attrition_output_accuracy: 0.7966 - val_attrition_output_loss: 0.4872 - val_department_output_accuracy: 0.6737 - val_department_output_loss: 0.6137 - val_loss: 1.1174
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - attrition_output_accuracy: 0.8562 - attrition_output_loss: 0.3679 - department_output_accuracy: 0.6687 - department_output_loss: 0.6019 - loss: 0.9687 - val_attrition_output_accuracy: 0.8008 -

In [20]:
# Evaluate the model with the testing data
test_results = model.evaluate(
    X_test_scaled,
    {
        'department_output': y_test_dept_encoded,
        'attrition_output': y_test_attrition_encoded
    },
    verbose=1
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - attrition_output_accuracy: 0.8447 - attrition_output_loss: 5.8321 - department_output_accuracy: 0.6917 - department_output_loss: 805.9984 - loss: 815.4077 


In [21]:
# Print the accuracy for both department and attrition
# Print the accuracy for both department and attrition
print(f"Attrition predictions accuracy: {test_results[1]}")
print(f"Department predictions accuracy: {test_results[2]}")

Attrition predictions accuracy: 721.9269409179688
Department predictions accuracy: 4.969839096069336


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1. Accuracy may not be the best metric for this data, especially for attrition prediction. Employee attrition is typically an imbalanced problem where most employees stay (majority class) and fewer leave (minority class). In such cases, accuracy can be misleading because a model could achieve high accuracy by simply predicting that no one will leave. Better metrics would include precision, recall, F1-score, or AUC-ROC, which provide more insight into how well the model identifies the minority class (employees who will leave). For department prediction, accuracy is more appropriate since departments are likely more balanced.
2. I chose softmax for the department output layer and sigmoid for the attrition output layer. Softmax is ideal for multi-class classification (predicting which of several departments an employee belongs to) because it outputs probabilities that sum to 1 across all classes. Sigmoid is perfect for binary classification (predicting whether an employee will leave or stay) because it outputs a probability between 0 and 1 for a single class. These activation functions match the mathematical requirements of their respective prediction tasks.
3. Several improvements could enhance this model: (1) Feature engineering - adding derived features like tenure ratios, salary-to-market comparisons, or interaction terms between existing features; (2) Handle class imbalance - using techniques like SMOTE, class weights, or stratified sampling to better handle imbalanced attrition data; (3) Hyperparameter tuning - optimizing learning rate, batch size, network architecture, and dropout rates through grid search or random search; (4) Regularization - adding dropout layers or L1/L2 regularization to prevent overfitting; (5) Advanced architectures - experimenting with different network depths, layer sizes, or even ensemble methods to improve prediction performance.